In [7]:
import os
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

# =========================================================
# 1. Dynamic Repo Path Setup (Works on GitHub & Locally)
# =========================================================
BASE_DIR = Path.cwd().parent if Path.cwd().name == "Python" else Path.cwd()
DATA_FOLDER = BASE_DIR / "Data"

# Database Configuration
MYSQL_USER = os.getenv("DB_USER", "root")
MYSQL_PASSWORD = os.getenv("DB_PASS", "password")  # Set your MySQL password here
MYSQL_HOST = os.getenv("DB_HOST", "localhost")
MYSQL_PORT = os.getenv("DB_PORT", "3306")
DATABASE_NAME = os.getenv("DB_NAME", "shopsphere")

CONNECTION_STR = f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{DATABASE_NAME}"
engine = create_engine(CONNECTION_STR)

# =========================================================
# 2. File Ingestion Configuration (In Dependency Order)
# =========================================================
FILES_CONFIG = [
    {"filename": "products.csv", "table": "products", "dates": [], "primary_key": "product_id"},
    {"filename": "customers.csv", "table": "customers", "dates": ["signup_date"], "primary_key": "customer_id"},
    {"filename": "orders.csv", "table": "orders", "dates": ["order_date"], "primary_key": "order_id"},
    {"filename": "order_items.csv", "table": "order_items", "dates": [], "primary_key": "order_item_id"},
    {"filename": "payments.csv", "table": "payments", "dates": ["payment_date"], "primary_key": "payment_id"},
    {"filename": "returns.csv", "table": "returns", "dates": ["return_date"], "primary_key": "return_id"},
]

# =========================================================
# 3. Batch Ingestion Pipeline
# =========================================================
def run_ingestion():
    if not DATA_FOLDER.exists():
        print(f"[ERROR] Data folder not found at: {DATA_FOLDER}")
        return

    with engine.begin() as conn:
        for item in FILES_CONFIG:
            file_path = DATA_FOLDER / item["filename"]
            table_name = item["table"]
            pk = item["primary_key"]

            if not file_path.exists():
                print(f"[ERROR] File not found: {file_path}")
                continue

            print(f"Reading {item['filename']}...")
            df = pd.read_csv(file_path, parse_dates=item["dates"])

            # Remove duplicate primary keys from the CSV if any exist
            if pk and pk in df.columns:
                initial_count = len(df)
                df = df.drop_duplicates(subset=[pk], keep="first")
                dropped = initial_count - len(df)
                if dropped > 0:
                    print(f"  [CLEAN] Dropped {dropped} duplicate rows based on '{pk}'.")

            # Wipe existing table data
            conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
            conn.execute(text(f"TRUNCATE TABLE {table_name};"))
            conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

            print(f"Ingesting into '{table_name}'...")
            df.to_sql(
                name=table_name,
                con=conn,
                if_exists="append",
                index=False,
                chunksize=5000,
                method="multi",
            )
            print(f"  [SUCCESS] {len(df):,} rows inserted into '{table_name}'.")

    print("\nAll files successfully processed into MySQL database!")

if __name__ == "__main__":
    run_ingestion()

Reading products.csv...
Ingesting into 'products'...
  [SUCCESS] 480 rows inserted into 'products'.
Reading customers.csv...
  [CLEAN] Dropped 30 duplicate rows based on 'customer_id'.
Ingesting into 'customers'...
  [SUCCESS] 15,000 rows inserted into 'customers'.
Reading orders.csv...
Ingesting into 'orders'...
  [SUCCESS] 100,000 rows inserted into 'orders'.
Reading order_items.csv...
Ingesting into 'order_items'...
  [SUCCESS] 163,869 rows inserted into 'order_items'.
Reading payments.csv...
Ingesting into 'payments'...
  [SUCCESS] 100,000 rows inserted into 'payments'.
Reading returns.csv...
Ingesting into 'returns'...
  [SUCCESS] 9,869 rows inserted into 'returns'.

All files successfully processed into MySQL database!
